# Time Series Forecasting Template

This notebook is a cleaned demo template for a simple daily time-series forecasting project.

It does **not** contain private or company-specific data. Replace the demo dataset with your own CSV or Parquet file when you are ready.

## Project workflow
1. Load or create a demo dataset
2. Prepare date and target columns
3. Create simple time-series features
4. Train a forecasting model
5. Make predictions
6. Evaluate model quality
7. Visualize actual vs predicted values


## 1. Import libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


## 2. Load data

For demo purposes, this notebook creates a synthetic daily dataset.

To use your own file later, replace this cell with:

```python
df = pd.read_csv("your_file.csv")
# or
df = pd.read_parquet("your_file.parquet")
```

Expected columns:
- `date`: daily date column
- `target`: numeric value to forecast


In [ ]:
# Create demo data without using any real company data
np.random.seed(42)

dates = pd.date_range(start="2020-01-01", end="2022-12-31", freq="D")
trend = np.linspace(100, 180, len(dates))
weekly_pattern = 15 * np.sin(2 * np.pi * dates.dayofweek / 7)
noise = np.random.normal(0, 8, len(dates))

# Generic target variable for forecasting
df = pd.DataFrame({
    "date": dates,
    "target": trend + weekly_pattern + noise
})

print("Date range:", df["date"].min(), "→", df["date"].max())
print("Rows:", len(df))
df.head()


## 3. Quick data check


In [ ]:
df.info()


In [ ]:
df.describe()


## 4. Plot the target variable


In [ ]:
df.iloc[:150].plot(x="date", y="target", title="Demo target variable: first 150 days")
plt.xlabel("Date")
plt.ylabel("Target")
plt.tight_layout()
plt.show()


## 5. Create time-series features

Features used in this demo:
- day of week
- month
- lag 1: previous day value
- lag 7: value from the same weekday one week before
- rolling mean over the previous 7 days


In [ ]:
df = df.sort_values("date").reset_index(drop=True)

df["day_of_week"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["lag_1"] = df["target"].shift(1)
df["lag_7"] = df["target"].shift(7)
df["rolling_mean_7"] = df["target"].shift(1).rolling(window=7).mean()

# Remove rows where lag features are not available
df_model = df.dropna().reset_index(drop=True)

FEATURES = ["day_of_week", "month", "lag_1", "lag_7", "rolling_mean_7"]
TARGET = "target"

print("Feature columns:", FEATURES)
df_model.head()


## 6. Train/test split

The model is trained on historical data and tested on the final part of the timeline.


In [ ]:
split_date = pd.Timestamp("2022-01-01")

train = df_model[df_model["date"] < split_date].copy()
test = df_model[df_model["date"] >= split_date].copy()

X_train = train[FEATURES]
y_train = train[TARGET]
X_test = test[FEATURES]
y_test = test[TARGET]

print("Training rows:", len(train))
print("Testing rows:", len(test))


## 7. Train model


In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    min_samples_leaf=3
)

model.fit(X_train, y_train)
print("Model trained successfully.")


## 8. Make predictions


In [ ]:
test["prediction"] = model.predict(X_test)

test[["date", "target", "prediction"]].head()


## 9. Evaluate model


In [ ]:
mae = mean_absolute_error(y_test, test["prediction"])
rmse = mean_squared_error(y_test, test["prediction"], squared=False)

metrics = {
    "MAE": round(mae, 2),
    "RMSE": round(rmse, 2),
    "n_test_rows": len(test),
    "first_test_date": str(test["date"].min().date()),
    "last_test_date": str(test["date"].max().date())
}

metrics


## 10. Visualize actual vs predicted values


In [ ]:
test.iloc[:150].plot(
    x="date",
    y=["target", "prediction"],
    title="Actual vs Predicted Values"
)
plt.xlabel("Date")
plt.ylabel("Value")
plt.tight_layout()
plt.show()


## 11. Plot prediction error


In [ ]:
test["absolute_error"] = np.abs(test["target"] - test["prediction"])

test.plot(x="date", y="absolute_error", title="Absolute Error Over Time")
plt.xlabel("Date")
plt.ylabel("Absolute Error")
plt.tight_layout()
plt.show()


## Notes

This is a portfolio-safe template. It contains no private store IDs, no supermarket-specific columns, and no original course/company text.

Possible improvements:
- Add more lag features
- Try another model
- Add cross-validation for time series
- Compare several forecasting approaches
- Add a short written conclusion
